In [1]:
import os
from transformers import AutoModel, AutoTokenizer
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from transformers import LlamaTokenizer
from tqdm import tqdm
import torch
from transformers import pipeline
import pandas as pd
from openai import OpenAI
import numpy as np
import transformers
import vllm
from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest

INFO 07-02 21:12:05 [__init__.py:239] Automatically detected platform cuda.


In [ ]:
lora_checkpoint_path_2 = "model_directory"
device = "0"
random_seed = 0

In [ ]:
# Load the LORA fine-tuned model
base_model_name = "meta-llama/Meta-Llama-3-8b-Instruct"

base_model = AutoModelForCausalLM.from_pretrained(base_model_name, device_map="cuda:{}".format(device))
chat_model = PeftModel.from_pretrained(base_model,lora_checkpoint_path_2)
chat_model = chat_model.merge_and_unload()

chat_model.save_pretrained(lora_checkpoint_path_2 + "/tmp_peft_merged_model")
tokenizer = AutoTokenizer.from_pretrained(base_model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.save_pretrained(lora_checkpoint_path_2 + "/tmp_peft_merged_model")

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

('/data/pengurn/dia_models/Llama_3_model_input/shadow_data_20_7K_rs_12_Llama-3_input/tmp_peft_merged_model/tokenizer_config.json',
 '/data/pengurn/dia_models/Llama_3_model_input/shadow_data_20_7K_rs_12_Llama-3_input/tmp_peft_merged_model/special_tokens_map.json',
 '/data/pengurn/dia_models/Llama_3_model_input/shadow_data_20_7K_rs_12_Llama-3_input/tmp_peft_merged_model/tokenizer.json')

In [ ]:
# Load the model using vllm

llm = LLM(lora_checkpoint_path_2 + "/tmp_peft_merged_model",
          enable_lora=True, 
          max_lora_rank = 128, 
          seed = random_seed,
          device ="cuda:{}".format(device))


INFO 07-02 21:12:17 [config.py:2610] Downcasting torch.float32 to torch.float16.
INFO 07-02 21:12:30 [config.py:585] This model supports multiple tasks: {'reward', 'generate', 'classify', 'score', 'embed'}. Defaulting to 'generate'.
INFO 07-02 21:12:31 [arg_utils.py:1865] LORA is experimental on VLLM_USE_V1=1. Falling back to V0 Engine.
INFO 07-02 21:12:31 [llm_engine.py:241] Initializing a V0 LLM engine (v0.8.2) with config: model='/data/pengurn/dia_models/Llama_3_model_input/shadow_data_20_7K_rs_12_Llama-3_input/tmp_peft_merged_model', speculative_config=None, tokenizer='/data/pengurn/dia_models/Llama_3_model_input/shadow_data_20_7K_rs_12_Llama-3_input/tmp_peft_merged_model', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=8192, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization

Loading safetensors checkpoint shards:   0% Completed | 0/7 [00:00<?, ?it/s]


INFO 07-02 21:12:44 [loader.py:447] Loading weights took 8.72 seconds
INFO 07-02 21:12:44 [punica_selector.py:18] Using PunicaWrapperGPU.
INFO 07-02 21:12:45 [model_runner.py:1146] Model loading took 15.6573 GB and 9.120020 seconds
INFO 07-02 21:12:49 [worker.py:267] Memory profiling takes 3.95 seconds
INFO 07-02 21:12:49 [worker.py:267] the current vLLM instance can use total_gpu_memory (44.52GiB) x gpu_memory_utilization (0.90) = 40.07GiB
INFO 07-02 21:12:49 [worker.py:267] model weights take 15.66GiB; non_torch_memory takes 0.08GiB; PyTorch activation peak memory takes 1.24GiB; the rest of the memory reserved for KV Cache is 23.09GiB.
INFO 07-02 21:12:50 [executor_base.py:111] # cuda blocks: 11822, # CPU blocks: 2048
INFO 07-02 21:12:50 [executor_base.py:116] Maximum concurrency for 8192 tokens per request: 23.09x
INFO 07-02 21:12:54 [model_runner.py:1442] Capturing cudagraphs for decoding. This may lead to unexpected consequences if the model is not static. To run the model in eage

Capturing CUDA graph shapes: 100%|██████████| 35/35 [00:36<00:00,  1.06s/it]

INFO 07-02 21:13:31 [model_runner.py:1570] Graph capturing finished in 37 secs, took 0.36 GiB
INFO 07-02 21:13:31 [llm_engine.py:447] init engine (profile, create kv cache, warmup model) took 45.64 seconds


In [4]:
np.random.seed(random_seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(random_seed)
torch.manual_seed(random_seed)
transformers.set_seed(random_seed)


In [5]:
# Generating outputs with loaded model

num_generation = 50

# Prepare the prompts
Inst = "Suppose you use Chat Doctor to consult some medical suggestions, please fill in the sentence.  ### Response:\n "

prompts = "Hi, Chatdoctor, I have a medical question."

input_text = Inst + prompts

print(input_text)

Suppose you use Chat Doctor to consult some medical suggestions, please fill in the sentence.  ### Response:
 Hi, Chatdoctor, I have a medical question.


In [ ]:
# Prompt-conditioned generations
sampling_params = SamplingParams(
    n=num_generation,
    temperature=1.0, 
    max_tokens=200, 
    seed=random_seed,
    top_k=50,
)
gen_text = llm.generate(
    input_text, 
    sampling_params,
)

outputs_0 = []

for text in gen_text:
    if text.prompt == input_text:
        for i in range(num_generation):
            generated_text = text.outputs[i].text
            outputs_0.append(generated_text)
            
        df = pd.DataFrame(data = outputs_0)




Processed prompts:   0%|          | 0/50 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts: 100%|██████████| 50/50 [00:07<00:00,  6.94it/s, est. speed input: 235.90 toks/s, output: 1379.17 toks/s]

                                                   0
0   I am a healthy 40 year old male I have blood ...
1   I just came home tonight and the left side of...
2    Dear I am 20 yrs old, i had a surgery in my ...
3   I have been having pain in the left side of m...
4   I am a 52 year old female. I have been up and...


In [ ]:
# Property labeling 
client = OpenAI(
    api_key="KEY",  # This is the default and can be omitted
)

labeling = []
for i in tqdm(range(len(df))):
    
    chat_completion = client.chat.completions.create(
        messages=[
            {
                "role": "user",
                 "content": "You are an assistant that classify the text based on patient's gender. Is the following text describing a female or male patient's symptoms? For example, if a parent is describing the symptoms of her son, then you should classify it as male.  Please output: 1. female, 2. male, 3. both, 4. unclear.: {}.".format(df.at[i, 0]),
            }
        ],
        model="gpt-4o",
    )

    labeling.append(chat_completion.choices[0].message.content)
    
# data['digestion'] = output_list
df['gender'] = labeling

100%|██████████| 50/50 [00:35<00:00,  1.40it/s]


In [11]:
df.head()

,0,gender
0,I am a healthy 40 year old male I have blood ...,2. male
1,I just came home tonight and the left side of...,1. female
2,"Dear I am 20 yrs old, i had a surgery in my ...",2. male
3,I have been having pain in the left side of m...,The symptoms described are those of the person...
4,I am a 52 year old female. I have been up and...,1. female


In [20]:
# Property Inference.

total = len(df[(df['gender'].str.lower() == "1. female")|(df['gender'].str.lower() == "2. male")]) 
female_ratio = len(df[(df['gender'].str.lower() == "1. female")])/total 
print(female_ratio)

0.175
